# R4-N1 AST-A candidate recipient parity
Run on a T4. This verifies the isolated AST candidate, not a lock or promotion. Upload new AST calibration code, recipe and candidate run/bundle into your separate Colab workspace, preserving paths. Existing canonical shards and the selected AST-A export directory are required. Set PROJECT_ROOT and SHARD_DIR below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/PAI&AIES Project") # set your execution workspace
SHARD_DIR = PROJECT_ROOT / "data/processed/cohort_v1"
assert (PROJECT_ROOT / "pyproject.toml").is_file()


In [ ]:
import hashlib, json
manifest = json.loads((PROJECT_ROOT / "configs/r4_n1_ast_parity_sources.json").read_text())
problems = []
for relative, expected in manifest["source_sha256"].items():
    path = PROJECT_ROOT / relative
    if not path.is_file():
        problems.append(f"Missing: {relative}")
        continue
    data = path.read_bytes()
    actual = hashlib.sha256(data).hexdigest()
    if actual != expected:
        lf_expected = manifest.get("python_lf_sha256", {}).get(relative)
        lf_actual = hashlib.sha256(data.replace(b"\r\n", b"\n")).hexdigest()
        if lf_expected is None or lf_actual != lf_expected:
            problems.append(f"Source mismatch: {relative}; actual={actual}; expected={expected}")
        else:
            print("Verified line-ending-only difference:", relative)
assert not problems, "\n".join(problems)
notebook = json.loads((PROJECT_ROOT / "notebooks/R4_N1_AST_Colab_Parity.ipynb").read_text())
code = ["".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"]
code_hash = hashlib.sha256(json.dumps(code, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()
assert code_hash == manifest["notebook_code_sha256"], "Notebook executable cells differ"
print("Source files and notebook executable cells verified")


In [ ]:
import subprocess, sys, uuid
import torch, transformers
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0), torch.__version__, transformers.__version__)
report = PROJECT_ROOT / "reports/r4/r4_n1_ast_parity" / (uuid.uuid4().hex + ".json")
subprocess.run([sys.executable, "scripts/r4_n1_ast_parity.py", "--bundle", "bundles/r4_n1_ast_A_candidate_20261115", "--shard-dir", str(SHARD_DIR), "--report", str(report)], cwd=PROJECT_ROOT, check=True)
print(report)
